# Create Supervisor Agent (Product manuals: Genie + Knowledge Assistant)

This notebook creates a [Supervisor Agent](https://docs.databricks.com/api/workspace/supervisoragents) (**Beta**) that coordinates:

- The **Genie space** from `create_genie_space_productmanuals.ipynb` (title ``{table_prefix}_Power Tool Product Comparison``).
- The **Knowledge Assistant** from `create_or_update_knowledge_assistant_productmanuals.ipynb` (display name ``{table_prefix}-power-tool-manuals-assistant``).

**Prerequisites:** Run the product manuals Genie and Knowledge Assistant notebooks (or their job tasks) first.

**API:** [Supervisor Agents](https://docs.databricks.com/api/workspace/supervisoragents) · [Knowledge Assistants](https://docs.databricks.com/api/workspace/knowledgeassistants) · [Genie](https://docs.databricks.com/api/workspace/genie/createspace)

In [0]:
# %pip install databricks-sdk -q

import sys, os
_notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
_src_root = "/Workspace" + os.path.dirname(os.path.dirname(_notebook_path))
for _subdir in ("genie_space", "knowledge_assistant", "supervisor_agent"):
    _path = os.path.join(_src_root, _subdir)
    if _path not in sys.path:
        sys.path.insert(0, _path)

from databricks.sdk import WorkspaceClient

from manage_genie_space import get_existing_genie_space, print_space_info
from manage_knowledge_assistant import get_knowledge_assistant_id_by_display_name, get_knowledge_assistant
from manage_supervisor_agent import create_supervisor_agent, create_supervisor_tool

w = WorkspaceClient()

# Defaults match databricks_etl/databricks.yml (overridden by job base_parameters when scheduled)
dbutils.widgets.text("catalog", "nikks_fevm_workspace_7405607030687545")
dbutils.widgets.text("schema", "test1")
dbutils.widgets.text("table_prefix", "extract_1")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
table_prefix = dbutils.widgets.get("table_prefix")

GENIE_SPACE_TITLE = f"{table_prefix}_Power Tool Product Comparison"
KA_DISPLAY_NAME = f"{table_prefix}-power-tool-manuals-assistant"

print(f"catalog={catalog} schema={schema} table_prefix={table_prefix}")
print(f"GENIE_SPACE_TITLE={GENIE_SPACE_TITLE}")
print(f"KA_DISPLAY_NAME={KA_DISPLAY_NAME}")

In [0]:
genie_space = get_existing_genie_space(w, GENIE_SPACE_TITLE)
if not genie_space:
    raise RuntimeError(
        f'Genie space titled "{GENIE_SPACE_TITLE}" was not found. '
        "Run create_genie_space_productmanuals.ipynb (or the Genie job task) first."
    )

print_space_info(genie_space)
genie_space_id = genie_space.get("space_id")
if not genie_space_id:
    raise RuntimeError(f"Genie response missing space_id: {genie_space!r}")

In [0]:
ka_id = get_knowledge_assistant_id_by_display_name(w, KA_DISPLAY_NAME)
if not ka_id:
    raise RuntimeError(
        f'Knowledge Assistant with display name "{KA_DISPLAY_NAME}" was not found. '
        "Run create_or_update_knowledge_assistant_productmanuals.ipynb (or the KA job task) first."
    )

ka = get_knowledge_assistant(w, ka_id)

In [0]:
ka

In [0]:
ka_id = get_knowledge_assistant_id_by_display_name(w, KA_DISPLAY_NAME)
if not ka_id:
    raise RuntimeError(
        f'Knowledge Assistant with display name "{KA_DISPLAY_NAME}" was not found. '
        "Run create_or_update_knowledge_assistant_productmanuals.ipynb (or the KA job task) first."
    )

ka = get_knowledge_assistant(w, ka_id)
ka_serving_endpoint_name = ka.get("endpoint_name")
if not ka_serving_endpoint_name:
    raise RuntimeError(
        "Could not determine serving_endpoint_name from get_knowledge_assistant response. "
        "Set the serving_endpoint_name widget (copy from the Knowledge Assistant / Agents UI). "
        f"Assistant keys (for debugging): {sorted(ka.keys())}"
    )

print(f"ka_id={ka_id}")
print(f"serving_endpoint_name={ka_serving_endpoint_name}")

In [0]:
supervisor_display_name = f"{table_prefix}-productmanuals-supervisor"
supervisor_description = (
    f"Supervisor for power-tool manuals: Genie on {table_prefix}_productmanuals_processed and "
    f"Knowledge Assistant ({KA_DISPLAY_NAME}) over manual PDFs."
)
supervisor_instructions = (
    "Delegate comparisons and structured specs to Genie and manual text / procedures to the Knowledge Assistant when appropriate."
)

agent = create_supervisor_agent(
    w,
    display_name=supervisor_display_name,
    description=supervisor_description,
    instructions=supervisor_instructions,
)
supervisor_agent_id = agent.get("supervisor_agent_id") or agent.get("id")
if not supervisor_agent_id:
    raise RuntimeError(f"Unexpected create_supervisor_agent response: {agent!r}")

print("Supervisor Agent created.")
print(f"supervisor_agent_id: {supervisor_agent_id}")
print(f"endpoint_name: {agent.get('endpoint_name')}")

tool_genie = create_supervisor_tool(
    w,
    supervisor_agent_id=supervisor_agent_id,
    tool_id="genie-productmanuals",
    description="Genie space for SQL and comparisons on processed product manual specs.",
    tool_type="genie_space",
    genie_space={"id": str(genie_space_id)},
)
print("Genie tool:", tool_genie.get("tool_id"), tool_genie.get("name"))

tool_ka = create_supervisor_tool(
    w,
    supervisor_agent_id=supervisor_agent_id,
    tool_id="ka-productmanuals",
    description="Knowledge Assistant grounded on product manual PDFs in UC volume.",
    tool_type="knowledge_assistant",
    knowledge_assistant={
        "knowledge_assistant_id": ka_id,
        "serving_endpoint_name": ka_serving_endpoint_name,
    },
)
print("Knowledge Assistant tool:", tool_ka.get("tool_id"), tool_ka.get("name"))

In [0]:
print("--- Summary ---")
print(f"Supervisor display name: {supervisor_display_name}")
print(f"supervisor_agent_id: {supervisor_agent_id}")
print(f"supervisor endpoint_name: {agent.get('endpoint_name')}")
print(f"Genie space_id: {genie_space_id}")
print(f"Knowledge Assistant id: {ka_id}")
print(f"KA serving_endpoint_name: {ka_serving_endpoint_name}")
print("Tool IDs: genie-productmanuals, ka-productmanuals")